In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, current_timestamp
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [0]:
events = spark.read.table("default.bronze_events")
features_df = spark.read.table("default.silver_user_features")

In [0]:
label_df = events.groupBy("user_id") \
    .agg(
        F.max(
            F.when(F.col("event_type") == "purchase", 1).otherwise(0)
        ).alias("purchased")
    )

In [0]:
training_data = features_df.join(label_df, "user_id", "left")
training_data = training_data.fillna({"purchased": 0})

In [0]:
class_counts = training_data.groupBy("purchased").count().collect()
count_dict = {row["purchased"]: row["count"] for row in class_counts}
total = sum(count_dict.values())

count_0 = count_dict.get(0, 1)
count_1 = count_dict.get(1, 1)

training_data = training_data.withColumn(
    "class_weight",
    F.when(F.col("purchased") == 1,
           total / (2 * count_1))
     .otherwise(total / (2 * count_0))
)

In [0]:
assembler = VectorAssembler(
    inputCols=["total_events", "total_spent", "avg_price"],
    outputCol="features"
)

training_data = assembler.transform(training_data)

train, test = training_data.randomSplit([0.8, 0.2], seed=42)

lr = LogisticRegression(
    featuresCol="features",
    labelCol="purchased",
    weightCol="class_weight",
    maxIter=20
)

model = lr.fit(train)

In [0]:
evaluator = BinaryClassificationEvaluator(
    labelCol="purchased",
    metricName="areaUnderROC"
)

predictions = model.transform(test)
auc = evaluator.evaluate(predictions)

print("Test AUC:", auc)

In [0]:
full_features = assembler.transform(features_df)

full_predictions = model.transform(full_features)

In [0]:
from pyspark.sql.functions import col, current_timestamp
from pyspark.ml.functions import vector_to_array

gold_predictions = full_predictions.select(
    "user_id",
    vector_to_array(col("probability"))[1].alias("purchase_probability"),
    "prediction"
).withColumn(
    "scoring_timestamp",
    current_timestamp()
)

In [0]:
gold_predictions.write \
    .mode("overwrite") \
    .saveAsTable("default.gold_user_purchase_predictions")

In [0]:
display(
    spark.read.table("default.gold_user_purchase_predictions")
)

In [0]:
top_users = gold_predictions.orderBy(
    col("purchase_probability").desc()
)

display(top_users.limit(10))